In [ ]:
import time
import subprocess
import pyautogui
import Quartz
from AppKit import NSWorkspace, NSScreen
import os

# Initialize global variables
draw_app = None
screen_width, screen_height = 0, 0
draw_window_position = (0, 0)
draw_window_size = (0, 0)


In [5]:

def get_screen_dimensions():
    """Get the dimensions of all connected screens"""
    global screen_width, screen_height
    
    # Get the main screen dimensions
    main_screen = NSScreen.mainScreen()
    if main_screen:
        screen_width = main_screen.frame().size.width
        screen_height = main_screen.frame().size.height
        print('Screen width with NSS Screen',screen_height, screen_width)
    else:
        # Fallback to pyautogui if NSScreen fails
        screen_width, screen_height = pyautogui.size()
    
    return screen_width, screen_height

# how to get the template_path
def wait_for_element(template_path, timeout=10, confidence=0.8):
    """Wait for an element to appear on screen before proceeding"""
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            location = pyautogui.locateOnScreen(template_path, confidence=confidence)
            if location:
                return location
        except Exception:
            pass
        time.sleep(0.5)
    
    raise TimeoutError(f"Element {template_path} not found after {timeout} seconds")


def focus_application(app_name):
    """Focus on a specific application by name"""
    apps = NSWorkspace.sharedWorkspace().runningApplications()
    for app in apps:
        if app_name.lower() in app.localizedName().lower():
            app.activateWithOptions_(Quartz.NSApplicationActivateIgnoringOtherApps)
            return True
    return False

def get_relative_coordinates(x, y):
    """Convert absolute coordinates to coordinates relative to the Draw window"""
    global draw_window_position, draw_window_size
    
    rel_x = x - draw_window_position[0]
    rel_y = y - draw_window_position[1]
    
    # Ensure coordinates are within the window
    rel_x = max(0, min(rel_x, draw_window_size[0]))
    rel_y = max(0, min(rel_y, draw_window_size[1]))
    
    return rel_x, rel_y



def scale_coordinates(x, y, target_width=1920, target_height=1080):
    """Scale coordinates based on target resolution vs actual screen resolution"""
    global screen_width, screen_height
    
    # Calculate scaling factors
    width_scale = screen_width / target_width
    height_scale = screen_height / target_height
    
    # Scale coordinates
    scaled_x = int(x * width_scale)
    scaled_y = int(y * height_scale)
    
    return scaled_x, scaled_y

In [6]:
get_screen_dimensions()

Screen width with NSS Screen 1080.0 2560.0


(2560.0, 1080.0)

In [12]:
subprocess.Popen(['open', '-a', 'LibreOffice'])
time.sleep(2)  # Wait for LibreOffice to start

if not focus_application("LibreOffice"):
    raise Exception("Could not focus on LibreOffice")
else:
    print("Libre office open")

pyautogui.hotkey('tab', interval=0.2)  # Move to document type selection
for _ in range(4):  # Tab to Draw (adjust if needed)
    pyautogui.hotkey('tab', interval=0.2)
pyautogui.hotkey('space')  # Select Draw

# Wait for Draw to open (look for toolbar or other static element)
# For production code, replace 'draw_toolbar.png' with an actual screenshot of a Draw element
# wait_for_element('draw_toolbar.png', timeout=10)
time.sleep(5)  # Fallback waiting time

# Maximize window
pyautogui.hotkey('command', 'option', 'm')
time.sleep(1)

Libre office open


In [15]:
# Focus on LibreOffice
# if not focus_application("LibreOffice"):
#     return {
#         "content": [
#             {
#                 "type": "text",
#                 "text": "LibreOffice Draw is not open. Please call open_libreoffice_draw first."
#             }
#         ]
#     }
subprocess.Popen(['open', '-a', 'LibreOffice'])

x1, y1, x2, y2 = 940,340,1200,600

# Scale coordinates based on screen resolution
x1, y1 = scale_coordinates(x1, y1)
x2, y2 = scale_coordinates(x2, y2)

# Wait for Draw to be ready (look for a static element)
time.sleep(1)

# Select Rectangle tool using keyboard shortcuts
pyautogui.hotkey('f4')  # Opens shape toolbar
time.sleep(0.5)

# Navigate to Rectangle (might need adjustments based on actual LibreOffice Draw interface)
pyautogui.hotkey('down')  # Navigate to rectangle
time.sleep(0.2)
pyautogui.hotkey('return')  # Select rectangle
time.sleep(0.5)

# Draw the rectangle
pyautogui.moveTo(x1, y1)
pyautogui.mouseDown()
pyautogui.moveTo(x2, y2)
pyautogui.mouseUp()

# Return to selection tool
pyautogui.hotkey('escape')


In [41]:
import time
import subprocess
import pyautogui
import Quartz
from AppKit import NSWorkspace

# Initialize global variables
draw_app = None

def focus_application(app_name):
    """Focus on a specific application by name"""
    apps = NSWorkspace.sharedWorkspace().runningApplications()
    for app in apps:
        if app_name.lower() in app.localizedName().lower():
            app.activateWithOptions_(Quartz.NSApplicationActivateIgnoringOtherApps)
            return True
    return False

def open_libreoffice():
    """Open LibreOffice Draw tool"""
    try:
        # Start LibreOffice
        subprocess.Popen(['open', '-a', 'LibreOffice'])
        time.sleep(2)  # Wait for LibreOffice to start
        
        
        # Focus on LibreOffice
        if not focus_application("LibreOffice"):
            raise Exception("Could not focus on LibreOffice")
        
        
        # Click "Draw" from the Start Center - using exact coordinates
        open_draw_x, open_draw_y = 150, 475  # Exact coordinates as specified
        pyautogui.click(open_draw_x, open_draw_y)
        
        time.sleep(3)
    except Exception as e:
        return {
            "content": [
                {
                    "type": "text",
                    "text": f"Error opening LibreOffice: {str(e)}"
                }
            ]
        }


def draw_rectangle(x1=1050, y1=375, x2=1350, y2=675):
    """Draw a rectangle in LibreOffice Draw from (x1,y1) to (x2,y2) using exact coordinates"""
    try:
        # Focus on LibreOffice
        if not focus_application("LibreOffice"):
            return {
                "content": [
                    {
                        "type": "text",
                        "text": "LibreOffice Draw is not open. Please call open_libreoffice_draw first."
                    }
                ]
            }
        

        # Click "Draw" from the Start Center - using exact coordinates
        draw_button_x, draw_button_y = 90, 247  # Exact coordinates as specified
        pyautogui.click(draw_button_x, draw_button_y)
        
        # Wait for Draw to open
        time.sleep(3)
        
        # Ensure we're in Draw and not in the start center
        time.sleep(1)
        
        # Click directly on Rectangle tool at the exact position (no scaling)
        rect_tool_x, rect_tool_y = 90, 247  # Exact coordinates for the rectangle tool
        
        # Move mouse to rectangle tool position and perform click
        print(f"Moving mouse to rectangle tool at ({rect_tool_x}, {rect_tool_y})")
        pyautogui.moveTo(rect_tool_x, rect_tool_y, duration=0.5)  # Slower movement to ensure accuracy
        pyautogui.click()  # Explicit click
        time.sleep(1)  # Give time for the tool to be selected
        
        # Now draw the rectangle using the exact coordinates provided
        print(f"Drawing rectangle from ({x1}, {y1}) to ({x2}, {y2})")
        pyautogui.moveTo(x1, y1, duration=0.5)  # Move to start position
        pyautogui.mouseDown()  # Press mouse down
        time.sleep(0.5)  # Small delay to ensure mouse down is registered
        pyautogui.moveTo(x2, y2, duration=0.5)  # Move to end position
        pyautogui.mouseUp()  # Release mouse
        time.sleep(0.5)  # Give time for the rectangle to be drawn

        
        
        # Return to selection tool
        # selection_tool_x, selection_tool_y = 60, 120  # Example coordinates for selection tool
        # pyautogui.click(selection_tool_x, selection_tool_y)
        
        return {
            "content": [
                {
                    "type": "text",
                    "text": f"Rectangle drawn from ({x1},{y1}) to ({x2},{y2})"
                }
            ]
        }
    except Exception as e:
        return {
            "content": [
                {
                    "type": "text",
                    "text": f"Error drawing rectangle: {str(e)}"
                }
            ]
        }

def enter_text_in_rectangle(text):
    """Enters the text at the desired position into the rectangle"""

    x_coord, y_coord = 1195,520

    pyautogui.moveTo(x_coord, y_coord)

    # Double click
    pyautogui.doubleClick()
    
    # Wait a bit for the click to register
    time.sleep(2)

    # text = "Hello, this is automated text entry!"
    
    # Type the text
    pyautogui.typewrite(text)



In [42]:
open_libreoffice()
draw_rectangle()
text = 'Text to be entered in libreoffice'
enter_text_in_rectangle(text)

Moving mouse to rectangle tool at (90, 247)
Drawing rectangle from (1050, 375) to (1350, 675)


In [ ]:

# Debug helper function
async def get_mouse_position():
    """Get and report current mouse position - useful for debugging"""
    try:
        current_position = pyautogui.position()
        return {
            "content": [
                {
                    "type": "text",
                    "text": f"Current mouse position: {current_position}"
                }
            ]
        }
    except Exception as e:
        return {
            "content": [
                {
                    "type": "text",
                    "text": f"Error getting mouse position: {str(e)}"
                }
            ]
        }

Moving mouse to rectangle tool at (90, 247)
Drawing rectangle from (1050, 375) to (1350, 675)


{'content': [{'type': 'text',
   'text': 'Rectangle drawn from (1050,375) to (1350,675)'}]}

In [21]:
subprocess.Popen(['open', '-a', 'LibreOffice'])

x1, y1, x2, y2 = 940,340,1200,600

draw_button_x, draw_button_y = 90, 247 # Approximate position, adjust as needed
pyautogui.click(draw_button_x, draw_button_y)


# Wait for Draw to open
# time.sleep(3)

# # Maximize window using Command+Option+M or by clicking the maximize button
# # Scale coordinates for maximize button
# maximize_button_x, maximize_button_y = scale_coordinates(1880, 30)  # Adjust as needed
# pyautogui.click(maximize_button_x, maximize_button_y)
# time.sleep(1)

# # Get the active window position and size
# draw_window_position = (0, 0)  # Top-left corner for maximized window
# draw_window_size = (screen_width, screen_height)  # Full screen size
        

In [ ]:

async def draw_rectangle(x1, y1, x2, y2):
    """Draw a rectangle in LibreOffice Draw from (x1,y1) to (x2,y2)"""
    global draw_app
    
    try:
        # Focus on LibreOffice
        if not focus_application("LibreOffice"):
            return {
                "content": [
                    {
                        "type": "text",
                        "text": "LibreOffice Draw is not open. Please call open_libreoffice_draw first."
                    }
                ]
            }
        
        # Scale coordinates based on screen resolution
        x1, y1 = scale_coordinates(x1, y1)
        x2, y2 = scale_coordinates(x2, y2)
        
        # Wait for Draw to be ready (look for a static element)
        time.sleep(1)
        
        # Select Rectangle tool using keyboard shortcuts
        pyautogui.hotkey('f4')  # Opens shape toolbar
        time.sleep(0.5)
        
        # Navigate to Rectangle (might need adjustments based on actual LibreOffice Draw interface)
        pyautogui.hotkey('down')  # Navigate to rectangle
        time.sleep(0.2)
        pyautogui.hotkey('return')  # Select rectangle
        time.sleep(0.5)
        
        # Draw the rectangle
        pyautogui.moveTo(x1, y1)
        pyautogui.mouseDown()
        pyautogui.moveTo(x2, y2)
        pyautogui.mouseUp()
        
        # Return to selection tool
        pyautogui.hotkey('escape')
        
        return {
            "content": [
                {
                    "type": "text",
                    "text": f"Rectangle drawn from ({x1},{y1}) to ({x2},{y2})"
                }
            ]
        }
    except Exception as e:
        return {
            "content": [
                {
                    "type": "text",
                    "text": f"Error drawing rectangle: {str(e)}"
                }
            ]
        }

In [4]:







async def open_libreoffice_draw():
    """Open LibreOffice Draw maximized"""
    global draw_app, draw_window_position, draw_window_size, screen_width, screen_height
    
    try:
        # Get screen dimensions
        get_screen_dimensions()
        
        # Start LibreOffice Draw
        subprocess.Popen(['open', '-a', 'LibreOffice'])
        time.sleep(2)  # Wait for LibreOffice to start
        
        # Focus on LibreOffice
        if not focus_application("LibreOffice"):
            raise Exception("Could not focus on LibreOffice")
        
        time.sleep(1)
        
        # Select "Draw" from the Start Center using keyboard
        pyautogui.hotkey('tab', interval=0.2)  # Move to document type selection
        for _ in range(4):  # Tab to Draw (adjust if needed)
            pyautogui.hotkey('tab', interval=0.2)
        pyautogui.hotkey('space')  # Select Draw
        
        # Wait for Draw to open (look for toolbar or other static element)
        # For production code, replace 'draw_toolbar.png' with an actual screenshot of a Draw element
        # wait_for_element('draw_toolbar.png', timeout=10)
        time.sleep(5)  # Fallback waiting time
        
        # Maximize window
        pyautogui.hotkey('command', 'option', 'm')
        time.sleep(1)
        
        # Get the active window position and size 
        # (In production, replace with actual window detection logic)
        draw_window_position = (0, 0)  # Top-left corner
        draw_window_size = (screen_width, screen_height)  # Full screen
        
        return {
            "content": [
                {
                    "type": "text",
                    "text": "LibreOffice Draw opened successfully and maximized"
                }
            ]
        }
    except Exception as e:
        return {
            "content": [
                {
                    "type": "text",
                    "text": f"Error opening LibreOffice Draw: {str(e)}"
                }
            ]
        }

async def draw_rectangle(x1, y1, x2, y2):
    """Draw a rectangle in LibreOffice Draw from (x1,y1) to (x2,y2)"""
    global draw_app
    
    try:
        # Focus on LibreOffice
        if not focus_application("LibreOffice"):
            return {
                "content": [
                    {
                        "type": "text",
                        "text": "LibreOffice Draw is not open. Please call open_libreoffice_draw first."
                    }
                ]
            }
        
        # Scale coordinates based on screen resolution
        x1, y1 = scale_coordinates(x1, y1)
        x2, y2 = scale_coordinates(x2, y2)
        
        # Wait for Draw to be ready (look for a static element)
        time.sleep(1)
        
        # Select Rectangle tool using keyboard shortcuts
        pyautogui.hotkey('f4')  # Opens shape toolbar
        time.sleep(0.5)
        
        # Navigate to Rectangle (might need adjustments based on actual LibreOffice Draw interface)
        pyautogui.hotkey('down')  # Navigate to rectangle
        time.sleep(0.2)
        pyautogui.hotkey('return')  # Select rectangle
        time.sleep(0.5)
        
        # Draw the rectangle
        pyautogui.moveTo(x1, y1)
        pyautogui.mouseDown()
        pyautogui.moveTo(x2, y2)
        pyautogui.mouseUp()
        
        # Return to selection tool
        pyautogui.hotkey('escape')
        
        return {
            "content": [
                {
                    "type": "text",
                    "text": f"Rectangle drawn from ({x1},{y1}) to ({x2},{y2})"
                }
            ]
        }
    except Exception as e:
        return {
            "content": [
                {
                    "type": "text",
                    "text": f"Error drawing rectangle: {str(e)}"
                }
            ]
        }

async def add_text_in_draw(text):
    """Add text in LibreOffice Draw"""
    try:
        # Focus on LibreOffice
        if not focus_application("LibreOffice"):
            return {
                "content": [
                    {
                        "type": "text",
                        "text": "LibreOffice Draw is not open. Please call open_libreoffice_draw first."
                    }
                ]
            }
        
        # Wait for Draw to be ready
        time.sleep(1)
        
        # Get coordinates for text placement (center of screen by default)
        x, y = scale_coordinates(screen_width // 2, screen_height // 2)
        
        # Select Text tool using keyboard shortcut
        pyautogui.hotkey('f2')  # Text tool shortcut in LibreOffice Draw
        time.sleep(0.5)
        
        # Click where to add text
        pyautogui.click(x, y)
        time.sleep(0.5)
        
        # Type the text
        pyautogui.write(text)
        time.sleep(0.5)
        
        # Finish text entry
        pyautogui.hotkey('escape')
        
        return {
            "content": [
                {
                    "type": "text",
                    "text": f"Text: '{text}' added successfully"
                }
            ]
        }
    except Exception as e:
        return {
            "content": [
                {
                    "type": "text",
                    "text": f"Error adding text: {str(e)}"
                }
            ]
        }

# Additional helper functions

async def save_document(filename):
    """Save the current LibreOffice Draw document"""
    try:
        # Focus on LibreOffice
        if not focus_application("LibreOffice"):
            return {
                "content": [
                    {
                        "type": "text",
                        "text": "LibreOffice Draw is not open. Please call open_libreoffice_draw first."
                    }
                ]
            }
        
        # Save using keyboard shortcut
        pyautogui.hotkey('command', 's')
        time.sleep(1)
        
        # Type filename
        pyautogui.write(filename)
        time.sleep(0.5)
        
        # Press Enter to save
        pyautogui.hotkey('return')
        time.sleep(1)
        
        return {
            "content": [
                {
                    "type": "text",
                    "text": f"Document saved as '{filename}'"
                }
            ]
        }
    except Exception as e:
        return {
            "content": [
                {
                    "type": "text", 
                    "text": f"Error saving document: {str(e)}"
                }
            ]
        }

# Initialize screen dimensions on module load
get_screen_dimensions()

(2560.0, 1080.0)

In [49]:
!pip3 install google

In [52]:
import os
from dotenv import load_dotenv
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client
import asyncio
import google.generativeai as genai
from concurrent.futures import TimeoutError
from functools import partial

# Load environment variables from .env file
load_dotenv()

# Access your API key and initialize Gemini client correctly
api_key = os.getenv("GEMINI_API_KEY")
client = genai.configure(api_key=os.environ["GEMINI_API_KEY"])

max_iterations = 3
last_response = None
iteration = 0
iteration_response = []

async def generate_with_timeout(client, prompt, timeout=10):
    """Generate content with a timeout"""
    print("Starting LLM generation...")
    try:
        # Convert the synchronous generate_content call to run in a thread
        loop = asyncio.get_event_loop()
        response = await asyncio.wait_for(
            loop.run_in_executor(
                None, 
                lambda: client.models.generate_content(
                    model="gemini-2.0-flash",
                    contents=prompt
                )
            ),
            timeout=timeout
        )
        print("LLM generation completed")
        return response
    except TimeoutError:
        print("LLM generation timed out!")
        raise
    except Exception as e:
        print(f"Error in LLM generation: {e}")
        raise

def reset_state():
    """Reset all global variables to their initial state"""
    global last_response, iteration, iteration_response
    last_response = None
    iteration = 0
    iteration_response = []

async def main():
    reset_state()  # Reset at the start of main
    print("Starting main execution...")
    try:
        # Create a single MCP server connection
        print("Establishing connection to MCP server...")
        server_params = StdioServerParameters(
            command="python",
            args=["mcp_server.py"]
        )

        async with stdio_client(server_params) as (read, write):
            print("Connection established, creating session...")
            async with ClientSession(read, write) as session:
                print("Session created, initializing...")
                await session.initialize()
                
                # Get available tools
                print("Requesting tool list...")
                tools_result = await session.list_tools()
                tools = tools_result.tools
                print(f"Successfully retrieved {len(tools)} tools")

                # Create system prompt with available tools
                print("Creating system prompt...")
                print(f"Number of tools: {len(tools)}")
                
                try:
                    # First, let's inspect what a tool object looks like
                    # if tools:
                    #     print(f"First tool properties: {dir(tools[0])}")
                    #     print(f"First tool example: {tools[0]}")
                    
                    tools_description = []
                    for i, tool in enumerate(tools):
                        try:
                            # Get tool properties
                            params = tool.inputSchema
                            desc = getattr(tool, 'description', 'No description available')
                            name = getattr(tool, 'name', f'tool_{i}')
                            
                            # Format the input schema in a more readable way
                            if 'properties' in params:
                                param_details = []
                                for param_name, param_info in params['properties'].items():
                                    param_type = param_info.get('type', 'unknown')
                                    param_details.append(f"{param_name}: {param_type}")
                                params_str = ', '.join(param_details)
                            else:
                                params_str = 'no parameters'

                            tool_desc = f"{i+1}. {name}({params_str}) - {desc}"
                            tools_description.append(tool_desc)
                            print(f"Added description for tool: {tool_desc}")
                        except Exception as e:
                            print(f"Error processing tool {i}: {e}")
                            tools_description.append(f"{i+1}. Error processing tool")
                    
                    tools_description = "\n".join(tools_description)
                    print("Successfully created tools description")
                except Exception as e:
                    print(f"Error creating tools description: {e}")
                    tools_description = "Error loading tools"
                
                print("Created system prompt...")
                
                system_prompt = f"""You are a math agent solving problems in iterations. You have access to various mathematical tools.

Available tools:
{tools_description}

You must respond with EXACTLY ONE line in one of these formats (no additional text):
1. For function calls:
   FUNCTION_CALL: function_name|param1|param2|...
   
2. For final answers:
   FINAL_ANSWER: [number]

Important:
- When a function returns multiple values, you need to process all of them
- Only give FINAL_ANSWER when you have completed all necessary calculations
- Do not repeat function calls with the same parameters

Examples:
- FUNCTION_CALL: add|5|3
- FUNCTION_CALL: strings_to_chars_to_int|INDIA
- FINAL_ANSWER: [42]

DO NOT include any explanations or additional text.
Your entire response should be a single line starting with either FUNCTION_CALL: or FINAL_ANSWER:"""

                query = """Find the ASCII values of characters in INDIA and then return sum of exponentials of those values. """
                print("Starting iteration loop...")
                
                # Use global iteration variables
                global iteration, last_response
                
                while iteration < max_iterations:
                    print(f"\n--- Iteration {iteration + 1} ---")
                    if last_response is None:
                        current_query = query
                    else:
                        current_query = current_query + "\n\n" + " ".join(iteration_response)
                        current_query = current_query + "  What should I do next?"

                    # Get model's response with timeout
                    print("Preparing to generate LLM response...")
                    prompt = f"{system_prompt}\n\nQuery: {current_query}"
                    try:
                        response = await generate_with_timeout(client, prompt)
                        response_text = response.text.strip()
                        print(f"LLM Response: {response_text}")
                        
                        # Find the FUNCTION_CALL line in the response
                        for line in response_text.split('\n'):
                            line = line.strip()
                            if line.startswith("FUNCTION_CALL:"):
                                response_text = line
                                break
                        
                    except Exception as e:
                        print(f"Failed to get LLM response: {e}")
                        break


                    if response_text.startswith("FUNCTION_CALL:"):
                        _, function_info = response_text.split(":", 1)
                        parts = [p.strip() for p in function_info.split("|")]
                        func_name, params = parts[0], parts[1:]
                        
                        print(f"\nDEBUG: Raw function info: {function_info}")
                        print(f"DEBUG: Split parts: {parts}")
                        print(f"DEBUG: Function name: {func_name}")
                        print(f"DEBUG: Raw parameters: {params}")
                        
                        try:
                            # Find the matching tool to get its input schema
                            tool = next((t for t in tools if t.name == func_name), None)
                            if not tool:
                                print(f"DEBUG: Available tools: {[t.name for t in tools]}")
                                raise ValueError(f"Unknown tool: {func_name}")

                            print(f"DEBUG: Found tool: {tool.name}")
                            print(f"DEBUG: Tool schema: {tool.inputSchema}")

                            # Prepare arguments according to the tool's input schema
                            arguments = {}
                            schema_properties = tool.inputSchema.get('properties', {})
                            print(f"DEBUG: Schema properties: {schema_properties}")

                            for param_name, param_info in schema_properties.items():
                                if not params:  # Check if we have enough parameters
                                    raise ValueError(f"Not enough parameters provided for {func_name}")
                                    
                                value = params.pop(0)  # Get and remove the first parameter
                                param_type = param_info.get('type', 'string')
                                
                                print(f"DEBUG: Converting parameter {param_name} with value {value} to type {param_type}")
                                
                                # Convert the value to the correct type based on the schema
                                if param_type == 'integer':
                                    arguments[param_name] = int(value)
                                elif param_type == 'number':
                                    arguments[param_name] = float(value)
                                elif param_type == 'array':
                                    # Handle array input
                                    if isinstance(value, str):
                                        value = value.strip('[]').split(',')
                                    arguments[param_name] = [int(x.strip()) for x in value]
                                else:
                                    arguments[param_name] = str(value)

                            print(f"DEBUG: Final arguments: {arguments}")
                            print(f"DEBUG: Calling tool {func_name}")
                            
                            result = await session.call_tool(func_name, arguments=arguments)
                            print(f"DEBUG: Raw result: {result}")
                            
                            # Get the full result content
                            if hasattr(result, 'content'):
                                print(f"DEBUG: Result has content attribute")
                                # Handle multiple content items
                                if isinstance(result.content, list):
                                    iteration_result = [
                                        item.text if hasattr(item, 'text') else str(item)
                                        for item in result.content
                                    ]
                                else:
                                    iteration_result = str(result.content)
                            else:
                                print(f"DEBUG: Result has no content attribute")
                                iteration_result = str(result)
                                
                            print(f"DEBUG: Final iteration result: {iteration_result}")
                            
                            # Format the response based on result type
                            if isinstance(iteration_result, list):
                                result_str = f"[{', '.join(iteration_result)}]"
                            else:
                                result_str = str(iteration_result)
                            
                            iteration_response.append(
                                f"In the {iteration + 1} iteration you called {func_name} with {arguments} parameters, "
                                f"and the function returned {result_str}."
                            )
                            last_response = iteration_result

                        except Exception as e:
                            print(f"DEBUG: Error details: {str(e)}")
                            print(f"DEBUG: Error type: {type(e)}")
                            import traceback
                            traceback.print_exc()
                            iteration_response.append(f"Error in iteration {iteration + 1}: {str(e)}")
                            break

                    elif response_text.startswith("FINAL_ANSWER:"):
                        print("\n=== Agent Execution Complete ===")
                        result = await session.call_tool("open_paint")
                        print(result.content[0].text)

                        # Wait longer for Paint to be fully maximized
                        await asyncio.sleep(1)

                        # Draw a rectangle
                        result = await session.call_tool(
                            "draw_rectangle",
                            arguments={
                                "x1": 780,
                                "y1": 380,
                                "x2": 1140,
                                "y2": 700
                            }
                        )
                        print(result.content[0].text)

                        # Draw rectangle and add text
                        result = await session.call_tool(
                            "add_text_in_paint",
                            arguments={
                                "text": response_text
                            }
                        )
                        print(result.content[0].text)
                        break

                    iteration += 1

    except Exception as e:
        print(f"Error in main execution: {e}")
        import traceback
        traceback.print_exc()
    finally:
        reset_state()  # Reset at the end of main

if __name__ == "__main__":
    asyncio.run(main())
    
    


RuntimeError: asyncio.run() cannot be called from a running event loop